# 12.02 · Dataset preparation

**Chapter:** Fine-tuning & Post-training · **Milestone:** M6

## Goals
- [ ] Know the formats: chat `messages`, prompt–completion, preference pairs; JSONL
- [ ] Understand chat templates and **loss masking** (train on the answer, not the prompt)
- [ ] Clean data: dedupe, filter, balance; split so the test measures **generalisation**, not memory
- [ ] Grow diversity with LLM-generated variants; store and version the dataset

## Resources
- [HF TRL – Dataset formats](https://huggingface.co/docs/trl/dataset_formats) — free
- [HF Datasets docs](https://huggingface.co/docs/datasets/) — free
- [LIMA: Less Is More for Alignment (data quality > quantity)](https://arxiv.org/abs/2305.11206) — free

In [ ]:
import json
import random
from collections import Counter

from transformers import AutoTokenizer

from aieng import DATA
from aieng.llm import client
from task import dataset, messages

FT = DATA / "ft"; FT.mkdir(exist_ok=True)

## 1. Formats
All JSONL, one example per line:
```jsonc
// conversational (chat) — most common for instruct models
{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}
// prompt-completion — loss only on the completion
{"prompt": [{"role": "user", ...}], "completion": [{"role": "assistant", ...}]}
// preference (DPO, 12.04)
{"prompt": [...], "chosen": [{"role": "assistant", ...}], "rejected": [{"role": "assistant", ...}]}
```

In [ ]:
ex = dataset(1, seed=5)[0]
print(json.dumps({"messages": messages(ex["text"], ex["target"])}, ensure_ascii=False, indent=1))

## 2. Chat templates and loss masking
The model sees one token sequence. With prompt–completion data, the trainer sets labels to `-100` (ignored) on prompt tokens, so the loss —
and learning — comes only from the answer. Training on the prompt too wastes capacity teaching the model to predict *user* text.

In [ ]:
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
prompt_ids = tok.apply_chat_template(messages(ex["text"]), add_generation_prompt=True)
full_ids = tok.apply_chat_template(messages(ex["text"], ex["target"]))
if not isinstance(prompt_ids, list):  # some transformers versions return a BatchEncoding
    prompt_ids, full_ids = prompt_ids["input_ids"], full_ids["input_ids"]
labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
print(tok.decode(full_ids))
print(f"{len(full_ids)} tokens, of which {sum(l != -100 for l in labels)} carry loss:", repr(tok.decode([t for t, l in zip(full_ids, labels) if l != -100])))

**Use the model's own chat template** for training *and* inference. A mismatch (training with one format, serving with another) silently ruins
fine-tunes; it's the #1 "my fine-tune got worse" bug.

## 3. Make the raw pool, then clean it

In [ ]:
raw = dataset(4000, seed=1)
print("raw:", len(raw))

# exact dedupe on the input text
seen, deduped = set(), []
for e in raw:
    if e["text"] not in seen:
        seen.add(e["text"]); deduped.append(e)
print("after exact dedupe:", len(deduped))

# filter: drop absurdly short/long inputs (real data has junk rows)
filtered = [e for e in deduped if 15 <= len(e["text"]) <= 200]
print("after length filter:", len(filtered))

# balance check: the label distribution you train on is the one you'll get
print("currency:", Counter(e["target"]["currency"] for e in filtered))
print("spanish share:", round(sum(any(w in e["text"] for w in ("Pagué", "Transferencia", "mercado", "cobraron")) for e in filtered) / len(filtered), 2))

Real-world cleaning also includes **near-duplicate** removal (MinHash, or embeddings + cosine > 0.95, 07.01), PII scrubbing (11.04), fixing
label errors (an LLM or a second annotator flags disagreements), and removing anything that overlaps your test set.

## 4. Split for generalisation: hold out whole templates
Random splits leak: the test set contains the *same templates* as training, so a model can score 100% by memorising patterns.
Hold out entire templates (and in real life: customers, time periods, document sources) to test what you actually care about.

In [ ]:
HELD_OUT_PREFIXES = ("Netflix charged me", "me cobraron")  # the last English and Spanish templates in task.py
in_dist = [e for e in filtered if not e["text"].startswith(HELD_OUT_PREFIXES)]
ood = [e for e in filtered if e["text"].startswith(HELD_OUT_PREFIXES)]
random.Random(0).shuffle(in_dist)
splits = {"train": in_dist[:1500], "valid": in_dist[1500:1600], "test_in": in_dist[1600:1700], "test_ood": ood[:100]}
for name, rows in splits.items():
    (FT / f"payments-{name}.jsonl").write_text("\n".join(json.dumps({"messages": messages(e["text"], e["target"]), "target": e["target"]}, ensure_ascii=False) for e in rows))
    print(f"{name:9} {len(rows):5} -> data/ft/payments-{name}.jsonl")

## 5. More diversity with an LLM
Templates make data cheap but repetitive. Ask an LLM for new **phrasings**; keep labels exact by having the LLM write templates (with
placeholders) rather than final examples. Validate every generated template before use.

In [ ]:
c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}
out = c.chat.completions.create(model=MODEL, temperature=0.9, max_tokens=300, **EXTRA, messages=[{"role": "user", "content":
    "Write 6 short, varied, realistic messages someone might send to log a payment (bank SMS, chat slang, email line, Spanish). "
    "Use the exact placeholders {amt} and {date} once each. One per line, no numbering."}]).choices[0].message.content
new_templates = [t.strip(" -•") for t in out.splitlines() if t.count("{amt}") == 1 and t.count("{date}") == 1 and len(t) < 160]
print(f"{len(new_templates)} valid new templates:")
for t in new_templates:
    print("  ", t)

## 6. Share & version
Push to the Hugging Face Hub as a (private) dataset; the Hub versions it with git. Runs only if `HF_TOKEN` is set.
```python
from datasets import load_dataset
ds = load_dataset("json", data_files={k: f"data/ft/payments-{k}.jsonl" for k in splits})
ds.push_to_hub("your-username/payments-extraction", private=True)
```

## How much data?
For narrow format/behaviour tasks with LoRA: a few hundred to a few thousand **clean** examples is often enough (12.03 will show it). Quality
and coverage of edge cases beat volume. For new skills or broad behaviour: tens of thousands+.

## Exercises
1. Add the LLM-generated templates to `task.py` (as a third list) and regenerate the splits. Does the OOD set get easier?
2. Add **negative** examples: messages with no payment ("thanks for lunch!") whose target is `{"date": null, "amount": null, "currency": null}`.
3. Write a near-duplicate filter with embeddings (`aieng.rag.embed`) and measure how many of the 4,000 examples it would remove at cosine 0.95.

In [ ]:
# your code here